# Bibliometric Co-occurrence Network Analysis (Web of Science Data)

This notebook builds and analyzes several co-occurrence networks from a Web of Science (WoS) bibliographic export:

1. **Country collaboration network**
2. **Co-authorship network**
3. **Co-affiliation (institution) network**
4. **Subject Category co-occurrence network**
5. **Keyword / Technology co-occurrence network**

For every network the pipeline is:

1. Build the weighted co-occurrence graph from the relevant DataFrame column and save it as GraphML.
2. (Where applicable) Extract the giant (largest connected) component and save it as GraphML.
3. Run community detection (Louvain), compute modularity, cliques, core/periphery split, and betweenness centrality, label the graph with community IDs, and save the final GraphML for visualization.

**Gephi workflow tip:** after downloading a `.graphml` output file, open it in Gephi, try different layout algorithms (ForceAtlas2 usually gives the best results for these networks), size nodes by degree centrality, and enable node labels from the bottom panel (adjusting font size/type as needed). Use the printed modularity, clique count, core/periphery counts, and top betweenness-centrality nodes in your report.

> **Note:** This notebook is a refactored, English-only, and bug-fixed version of the original. Duplicated boilerplate code (giant-component extraction and community/metric analysis, previously copy-pasted 5 times) has been consolidated into shared helper functions, and the Subject Category section now correctly reads/writes its own files instead of the previously mismatched `Funding` filenames.

## 0. Environment Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Convert the Web of Science Text Export to Excel

This code converts the plain-text file exported from the Web of Science website into an Excel file. The resulting columns include: `Abstract`, `Authors`, `C3` (raw affiliations), `DOI`, `Funding`, `Journal`, `Keywords`, `Year`, `Addresses`, `Title`, `Cite`, `Subject_Category`, and the derived `Countries` column.

In [ ]:
# --- Make the shared `techintel` package importable ---------------------------
# The WoS parsing / graph / metrics helpers used to be copy-pasted into each
# notebook. They now live in techintel/ at the repository root; this finds that
# root by walking up from the working directory.
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "techintel" / "__init__.py").exists():
            return candidate
    raise RuntimeError(
        "Could not find the repository root (a directory containing 'techintel/')."
    )


REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from techintel import wos

# `parse_wos_file`, `clean_c3_field`, `extract_countries`, `save_to_excel` and
# `process_wos_to_excel` now live in techintel/wos.py and are shared with the
# SDG notebook. Aliased here so the rest of this notebook reads unchanged.
parse_wos_file = wos.parse_wos_file
clean_c3_field = wos.clean_c3_field
extract_countries = wos.extract_countries
save_to_excel = wos.save_to_excel
process_wos_to_excel = wos.process_wos_to_excel

# NOTE: the old `university_systems` filter list shipped fully commented out, so
# it never ran. To enable it, pass it explicitly, e.g.
#   df['C3'] = df['C3'].apply(lambda v: clean_c3_field(v, drop=university_systems))

if __name__ == "__main__":
    wos_filepath = "/content/drive/MyDrive/HPC/merged.txt"
    output_filepath = "output_wos_file.xlsx"
    process_wos_to_excel(wos_filepath, output_filepath)


In [ ]:
import pandas as pd

excel_file = './output_wos_file.xlsx'
df = pd.read_excel(excel_file)

## 2. Shared Helper Functions

All five networks below share the same three-stage pipeline (build → giant component → community/metric analysis). To avoid the copy-paste bugs present in the original notebook (e.g. the Subject Category giant-component step accidentally pointing at a non-existent `Funding` network file), the logic is defined **once** here and reused by every section.

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

from techintel import communities, graphs

# --- Shared primitives (techintel/graphs.py and techintel/communities.py) -----
build_cooccurrence_graph = graphs.build_cooccurrence_graph
load_graph = graphs.load_graph
get_giant_component = graphs.get_giant_component
extract_and_save_giant_component = graphs.extract_and_save_giant_component
save_graph_with_communities = graphs.save_graph
label_communities = communities.label_communities
find_cliques = communities.find_cliques
detect_core_periphery = communities.detect_core_periphery
calculate_betweenness_centrality = communities.betweenness_centrality

# `build_cooccurrence_graph` de-duplicates items *within* a row by default. The
# old local copy did not, so a row like "a; b; a" produced a self-loop `a-a` and
# counted `a-b` twice. Pass `dedupe=False` to reproduce the old (inflated) counts.


def calculate_modularity_and_communities(G):
    """Louvain partition plus its modularity."""
    partition = communities.louvain_partition(G)
    return partition, communities.partition_modularity(G, partition)


def detect_communities_greedy(G):
    """Alternative community detection: greedy modularity maximisation."""
    return list(nx.community.greedy_modularity_communities(G))


def analyze_graph(graphml_file, output_file, top_n=5):
    """Full pipeline: Louvain communities, modularity, cliques, core/periphery
    split, and betweenness centrality. Labels the graph with community IDs and
    saves the result.
    """
    G = load_graph(graphml_file, remove_self_loops=True)

    partition, modularity = calculate_modularity_and_communities(G)
    print(f"Modularity: {modularity}")
    print(f"Communities detected (using Louvain): {len(set(partition.values()))}")

    label_communities(G, partition)

    cliques = find_cliques(G)
    print(f"Number of cliques found: {len(cliques)}")

    core_nodes, periphery_nodes = detect_core_periphery(G)
    print(f"Core nodes: {len(core_nodes)}, Periphery nodes: {len(periphery_nodes)}")

    betweenness = calculate_betweenness_centrality(G)
    top_bridging_nodes = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:top_n]
    print(f"Top {top_n} bridging nodes based on betweenness centrality:")
    for node, value in top_bridging_nodes:
        print(f"Node: {node}, Betweenness Centrality: {value}")

    save_graph_with_communities(G, output_file)

    return {
        "graph": G,
        "partition": partition,
        "modularity": modularity,
        "cliques": cliques,
        "core_nodes": core_nodes,
        "periphery_nodes": periphery_nodes,
        "betweenness": betweenness,
    }


## 3. Co-Country Network Analysis

Builds a country collaboration network (nodes = countries, edges weighted by the number of co-authored papers between them) directly from the `Countries` column, then runs the shared community/metric analysis on it. No giant-component extraction step is used here, since the original pipeline analyzed the full country network directly.

After downloading `Countries_network_with_communities.graphml`, open it in Gephi, apply ForceAtlas2, size nodes by degree centrality, and enable labels for your report.

In [ ]:
G_countries = build_cooccurrence_graph(df, column='Countries', separator=';')
nx.write_graphml(G_countries, 'co_Countries_network.graphml')
print("Network saved as co_Countries_network.graphml")

country_results = analyze_graph(
    graphml_file='co_Countries_network.graphml',
    output_file='Countries_network_with_communities.graphml'
)

## 4. Co-Authorship Network Analysis

Builds the full co-authorship network from the pipe-separated `Authors` column, extracts its giant component, and then runs the shared community/metric analysis on the giant component.

After downloading the final `.graphml` file, open it in Gephi, apply ForceAtlas2, size nodes by degree centrality, and enable labels for your report.

In [ ]:
G_authors = build_cooccurrence_graph(df, column='Authors', separator='|')
nx.write_graphml(G_authors, 'Co-authorship_network.graphml')
print("Network saved as Co-authorship_network.graphml")

giant_authors = extract_and_save_giant_component(
    graphml_file='Co-authorship_network.graphml',
    output_file='giant_component_authorship_network.graphml'
)

authorship_results = analyze_graph(
    graphml_file='giant_component_authorship_network.graphml',
    output_file='giant_component_authorship_network_with_communities.graphml'
)

## 5. Co-Affiliation Network Analysis

Builds an institution (affiliation) co-occurrence network from the cleaned `C3` column, extracts the giant component, and runs the shared community/metric analysis on it.

After downloading the final `.graphml` file, open it in Gephi, apply ForceAtlas2, size nodes by degree centrality, and enable labels for your report.

In [ ]:
G_affiliations = build_cooccurrence_graph(df, column='C3', separator=';')
nx.write_graphml(G_affiliations, 'co_affiliation_network.graphml')
print("Network saved as co_affiliation_network.graphml")

giant_affiliations = extract_and_save_giant_component(
    graphml_file='co_affiliation_network.graphml',
    output_file='giant_component_affiliation_network.graphml'
)

affiliation_results = analyze_graph(
    graphml_file='giant_component_affiliation_network.graphml',
    output_file='giant_component_affiliation_network_with_communities.graphml'
)
print("end")

## 6. Subject Category Network Analysis

Builds a co-occurrence network of WoS Subject Categories from the `Subject_Category` column, extracts the giant component, and runs the shared community/metric analysis on it.

> **Bug fix:** in the original notebook, the giant-component and community-analysis cells for this section mistakenly referenced a `co_Funding_network.graphml` file that is never created anywhere else in the notebook (a leftover from copy-pasting the previous section). This version correctly builds, extracts, and analyzes the **Subject Category** network end-to-end.

After downloading the final `.graphml` file, open it in Gephi, apply ForceAtlas2, size nodes by degree centrality, and enable labels for your report.

In [ ]:
G_subject_category = build_cooccurrence_graph(df, column='Subject_Category', separator=';')
nx.write_graphml(G_subject_category, 'co_Subject_Category_network.graphml')
print("Network saved as co_Subject_Category_network.graphml")

giant_subject_category = extract_and_save_giant_component(
    graphml_file='co_Subject_Category_network.graphml',
    output_file='giant_component_Subject_Category_network.graphml'
)

subject_category_results = analyze_graph(
    graphml_file='giant_component_Subject_Category_network.graphml',
    output_file='giant_component_Subject_Category_network_with_communities.graphml'
)
print("end")

## 7. Keyword / Technology Network Analysis

This section cleans and lemmatizes the raw `Keywords` column (removing parenthetical content, stopwords, standalone abbreviations, and numeric tokens using spaCy + NLTK), builds a keyword co-occurrence network, and tags each node as `technology` or `general` based on a supplied technology term list (`technologies.txt`). It then extracts the giant component and runs the shared community/metric analysis.

> **Consistency fix:** the original notebook's final community-analysis cell for this section re-analyzed the *full* keyword graph instead of its giant component (inconsistent with every other section in the notebook). This version analyzes the **giant component**, matching the pattern used everywhere else.

After downloading the final `.graphml` file, open it in Gephi, apply ForceAtlas2, size nodes by degree centrality, and enable labels for your report.

In [ ]:
import re
import spacy
import nltk
import matplotlib.pyplot as plt
from nltk.corpus import stopwords

nlp = spacy.load("en_core_web_sm")
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

with open('/content/drive/MyDrive/SNA/technologies.txt', 'r') as f:
    technology_list = [line.strip().lower() for line in f]


def preprocess_keywords(text):
    """Clean and lemmatize a semicolon-separated raw keyword string into a de-duplicated list of terms."""
    if not isinstance(text, str):
        return []
    # Step 1: Remove anything inside parentheses (e.g., "machine learning (ml)" -> "machine learning")
    text = re.sub(r'\([^)]*\)', '', text).strip()

    # Step 2: Split by ';' separator to get individual keywords
    keywords = text.split(';')

    cleaned_keywords = []

    for keyword in keywords:
        keyword = keyword.lower().strip()  # Convert to lowercase and remove extra spaces

        # Step 3: Remove stopwords
        keyword_tokens = [word for word in keyword.split() if word not in stop_words]

        # Step 4: Remove standalone abbreviations (e.g., "CNN")
        keyword_cleaned = re.sub(r'\b[A-Z]{2,}\b', '', ' '.join(keyword_tokens)).strip()

        # Step 5: Lemmatize the words to convert plural to singular
        doc = nlp(keyword_cleaned)
        keyword_lemmatized = ' '.join([token.lemma_ for token in doc])

        # Step 6: Keep non-empty keywords (numeric keywords will be removed after the network creation)
        if keyword_lemmatized and not re.search(r'\d', keyword_lemmatized):  # Ensure no numbers are present
            cleaned_keywords.append(keyword_lemmatized)

    return list(set(cleaned_keywords))  # Remove duplicate keywords


df['Processed Keywords'] = df['Keywords'].apply(preprocess_keywords)

G_keywords = nx.Graph()

for keywords in df['Processed Keywords']:
    keywords = [keyword for keyword in keywords if keyword and not re.search(r'\d', keyword)]

    for i in range(len(keywords)):
        for j in range(i + 1, len(keywords)):
            if G_keywords.has_edge(keywords[i], keywords[j]):
                G_keywords[keywords[i]][keywords[j]]['weight'] += 1
            else:
                G_keywords.add_edge(keywords[i], keywords[j], weight=1)

            G_keywords.nodes[keywords[i]]['type'] = 'technology' if keywords[i] in technology_list else 'general'
            G_keywords.nodes[keywords[j]]['type'] = 'technology' if keywords[j] in technology_list else 'general'

nodes_to_remove = [node for node in G_keywords.nodes() if node.isdigit() or node == '']
G_keywords.remove_nodes_from(nodes_to_remove)

plt.figure(figsize=(10, 8))

pos = nx.spring_layout(G_keywords, seed=42)

node_colors = ['lightblue' if G_keywords.nodes[node]['type'] == 'technology' else 'lightgreen' for node in G_keywords.nodes()]

nx.draw(G_keywords, pos, with_labels=True, node_size=500, node_color=node_colors, font_size=10, font_weight="bold", edge_color="gray")

plt.title("Keyword Network Graph with Technology Terms", fontsize=16)
plt.show()

nx.write_graphml(G_keywords, 'keywords_network_with_technologies.graphml')
print("Network saved as keywords_network_with_technologies.graphml")

In [ ]:
giant_keywords = extract_and_save_giant_component(
    graphml_file='keywords_network_with_technologies.graphml',
    output_file='giant_component_keywords_network_with_technologies.graphml'
)

keywords_results = analyze_graph(
    graphml_file='giant_component_keywords_network_with_technologies.graphml',
    output_file='giant_component_keywords_network_with_technologies_with_communities.graphml'
)
print("end")